In [1]:
import os
from datasets import load_dataset, DatasetDict, concatenate_datasets
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer,T5Tokenizer, AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq, Seq2SeqTrainingArguments, Seq2SeqTrainer

c:\Users\ndhie\OneDrive\Desktop\medical-bidirectional-translator\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Set up Dataset

In [2]:
data_files_en = {
    "train": "train.en.txt",
    "validation": "val.en.new.txt",
    "test": "test.en.new.txt"
}
data_files_vi = {
    "train": "train.vi.txt",
    "validation": "val.vi.new.txt",
    "test": "test.vi.new.txt"
}

en_data = load_dataset("nhuvo/MedEV", data_files=data_files_en)
vi_data = load_dataset("nhuvo/MedEV", data_files=data_files_vi)

en_data = en_data.rename_column("text", "en")
vi_data = vi_data.rename_column("text", "vi")

for split in ["train", "validation", "test"]:
    assert len(en_data[split]) == len(vi_data[split]), (
        f"{split} bị lệch số dòng: "
        f"en={len(en_data[split])}, vi={len(vi_data[split])}"
    )

med_dataset = DatasetDict({
    "train": concatenate_datasets([en_data['train'], vi_data['train']], axis=1),
    "validation": concatenate_datasets([en_data['validation'], vi_data['validation']], axis=1),
    "test": concatenate_datasets([en_data['test'], vi_data['test']], axis=1)
})

print("Kiểm tra data:", med_dataset['train'][0])

Kiểm tra data: {'en': 'To evaluate clinical, subclinical symptoms of patients with otitis media with effusion and V.a at otorhinolaryngology department – Thai Nguyen national hospital', 'vi': 'Nghiên cứu đặc điểm lâm sàng, cận lâm sàng bệnh nhân viêm tai ứ dịch trên viêm V.A tại Khoa Tai mũi họng - Bệnh viện Trung ương Thái Nguyên'}


In [3]:
import random

for split in ["train", "validation", "test"]:
    print(f"\n===== {split.upper()} SAMPLE CHECK =====")

    for idx in random.sample(range(len(med_dataset[split])), 5):
        print("IDX:", idx)
        print("EN:", med_dataset[split][idx]["en"])
        print("VI:", med_dataset[split][idx]["vi"])
        print()


===== TRAIN SAMPLE CHECK =====
IDX: 56126
EN: Most hypertensive outpatients say they will return or refer others.
VI: Hầu hết bệnh nhân điều trị ngoại trú tăng huyết áp cho ý kiến sẽ quay trở lại hoặc giới thiệu cho người khác.

IDX: 176724
EN: Effects of spinal - epidural spinal anesthesia incorporating TCI propofol sedation on respiration, circulation and undesirable effects in lower abdominal surgery in the elderly (O bjective 3) 4.4.1.
VI: Ảnh hưởng của gây tê TTS – NMC kết hợp an thần TCI propofol trên hô hấp, tuần hoàn và các tác dụng không mong muốn trong phẫu thuật bụng dưới ở người cao tuổi (mục tiêu 3) 4.4.1.

IDX: 300708
EN: If this trial fails, anorectal manometry with balloon expulsion should be done to identify pelvic floor disorders and dyssynergic defecation (2).
VI: Nếu thử nghiệm này thất bại, thì cần phài tiến hành đo áp lực hậu môn trực tràng bằng cách trục xuất bóng để xác định các tình trạng rối loạn ở đáy chậu và rối loạn hậu môn-trực tràng chức năng (2).

IDX: 

In [4]:
print(med_dataset)

DatasetDict({
    train: Dataset({
        features: ['en', 'vi'],
        num_rows: 340897
    })
    validation: Dataset({
        features: ['en', 'vi'],
        num_rows: 8939
    })
    test: Dataset({
        features: ['en', 'vi'],
        num_rows: 8960
    })
})


# Investigating Dataset

In [5]:
import pandas as pd
import re

def to_df(split):
    return pd.DataFrame({
        "en": med_dataset[split]["en"],
        "vi": med_dataset[split]["vi"],
    })

dfs = {s: to_df(s) for s in ["train", "validation", "test"]}

In [6]:
vi_pattern = re.compile(
    r"[àáảãạăằắẳẵặâầấẩẫậèéẻẽẹêềếểễệìíỉĩị"
    r"òóỏõọôồốổỗộơờớởỡợ"
    r"ùúủũụưừứửữự"
    r"ỳýỷỹỵđ]",
    re.IGNORECASE,
)

In [7]:
import unicodedata
import re
from datasets import Dataset, DatasetDict

def normalize_text(x):
    x = unicodedata.normalize("NFC", str(x))
    x = re.sub(r"\s+", " ", x).strip()
    return x

def clean_df(df, remove_identical_vi=True):
    df = df.copy()

    df["en"] = df["en"].map(normalize_text)
    df["vi"] = df["vi"].map(normalize_text)

    df = df.drop_duplicates(subset=["en", "vi"]).reset_index(drop=True)

    if remove_identical_vi:
        mask = (
            (df["en"] == df["vi"])
            &
            (df["en"].str.contains(vi_pattern, na=False))
        )

        print("Remove suspicious identical Vietnamese rows:", mask.sum())

        df = df[~mask].reset_index(drop=True)

    return df

In [8]:
train_df = clean_df(dfs["train"])
val_df = clean_df(dfs["validation"])
test_df = clean_df(dfs["test"])

Remove suspicious identical Vietnamese rows: 8
Remove suspicious identical Vietnamese rows: 0
Remove suspicious identical Vietnamese rows: 1


In [9]:
clean_dfs = {
    "train": train_df,
    "validation": val_df,
    "test": test_df,
}

In [10]:
# 1. Rỗng / thiếu / chỉ có khoảng trắng
for s, df in clean_dfs.items():
    empty_en = df["en"].str.strip().eq("").sum()
    empty_vi = df["vi"].str.strip().eq("").sum()
    print(f"[{s}] empty en={empty_en}, empty vi={empty_vi}")

[train] empty en=0, empty vi=0
[validation] empty en=0, empty vi=0
[test] empty en=0, empty vi=0


In [11]:
# 2. Trùng lặp NỘI BỘ từng split (cặp en-vi giống hệt nhau)
for s, df in clean_dfs.items():
    dup = df.duplicated(subset=["en", "vi"]).sum()
    print(f"[{s}] duplicated pairs: {dup}")

[train] duplicated pairs: 0
[validation] duplicated pairs: 0
[test] duplicated pairs: 0


In [12]:
# 3. LEAKAGE giữa các split — quan trọng nhất, ảnh hưởng trực tiếp BLEU
train_pairs = set(zip(clean_dfs["train"]["en"], clean_dfs["train"]["vi"]))
val_pairs   = set(zip(clean_dfs["validation"]["en"], clean_dfs["validation"]["vi"]))
test_pairs  = set(zip(clean_dfs["test"]["en"], clean_dfs["test"]["vi"]))

print("train ∩ val  :", len(train_pairs & val_pairs))
print("train ∩ test :", len(train_pairs & test_pairs))
print("val ∩ test   :", len(val_pairs & test_pairs))
print()

# leakage chỉ theo câu nguồn (en) - model có thể đã "học thuộc" câu này dù target khác bản dịch
train_en_only  = set(clean_dfs["train"]["en"])
test_en_only   = set(clean_dfs["test"]["en"])
print("train.en ∩ test.en (chỉ câu nguồn):", len(train_en_only & test_en_only))

train ∩ val  : 13
train ∩ test : 7
val ∩ test   : 0

train.en ∩ test.en (chỉ câu nguồn): 95


In [13]:
# 4. en == vi (dòng bị copy nhầm, không thực sự dịch)
for s, df in clean_dfs.items():
    same = (df["en"].str.strip() == df["vi"].str.strip()).sum()
    print(f"[{s}] en == vi identical rows: {same}")

[train] en == vi identical rows: 136
[validation] en == vi identical rows: 2
[test] en == vi identical rows: 1


In [14]:
# 5. Tỷ lệ độ dài ký tự bất thường (dấu hiệu misalignment)
for s, df in clean_dfs.items():
    ratio = df["vi"].str.len() / df["en"].str.len().clip(lower=1)
    outliers = ((ratio > 4) | (ratio < 0.25)).sum()
    print(f"[{s}] length-ratio outliers (vi/en char ratio): {outliers} / {len(df)}")
    print(df.loc[(ratio > 4) | (ratio < 0.25), ["en","vi"]].head(3))

[train] length-ratio outliers (vi/en char ratio): 121 / 340714
                                                     en  \
197                                            Methods.   
500                                             Method.   
3826  Using qualitative research and phenomenologica...   

                                        vi  
197      Đối tượng-Phương pháp nghiên cứu.  
500    Đối tượng - Phương pháp nghiên cứu.  
3826  Đối tượng và phương pháp nghiên cứu.  
[validation] length-ratio outliers (vi/en char ratio): 2 / 8939
                  en                                                 vi
2715         Missing  Tỷ lệ bệnh loãng xương và các yếu tố nguy cơ t...
8546  Medical Assoc.                                                TS.
[test] length-ratio outliers (vi/en char ratio): 2 / 8958
                                                     en  \
3263                                       Ec-ic bypass   
3837  Consist of perforated gastroduodenal ulcer pat...   

    

In [15]:
# 6. Kiểm tra cột "vi" có thực sự chứa tiếng Việt không (phát hiện lỗi ghép cột / lệch dòng)
for s, df in clean_dfs.items():
    no_diacritics = (~df["vi"].str.contains(vi_pattern)).sum()
    print(f"[{s}] dòng 'vi' KHÔNG có dấu tiếng Việt: {no_diacritics} / {len(df)}")

[train] dòng 'vi' KHÔNG có dấu tiếng Việt: 726 / 340714
[validation] dòng 'vi' KHÔNG có dấu tiếng Việt: 11 / 8939
[test] dòng 'vi' KHÔNG có dấu tiếng Việt: 9 / 8958


In [16]:
# 7. Phát hiện các dòng en == vi và thực chất là tiếng Việt
for s, df in clean_dfs.items():

    mislabeled = df[
        (df["en"].str.strip() == df["vi"].str.strip()) &
        (df["en"].str.contains(vi_pattern, na=False))
    ]

    print(f"\n[{s}] suspicious identical Vietnamese rows: {len(mislabeled)}")

    if len(mislabeled):
        print(mislabeled.head(20))


[train] suspicious identical Vietnamese rows: 0

[validation] suspicious identical Vietnamese rows: 0

[test] suspicious identical Vietnamese rows: 0


In [17]:
from datasets import Dataset

# 2. Tạo key cho leakage check
train_df["pair_key"] = train_df["en"] + "|||" + train_df["vi"]
val_keys = set(val_df["en"] + "|||" + val_df["vi"])
test_keys = set(test_df["en"] + "|||" + test_df["vi"])

leak_keys = val_keys | test_keys

print("Remove train leakage:", train_df["pair_key"].isin(leak_keys).sum())

train_df = train_df[
    ~train_df["pair_key"].isin(leak_keys)
].drop(columns=["pair_key"]).reset_index(drop=True)


# 3. Xóa dòng en == vi nhưng thực chất là tiếng Việt
mislabeled_mask = (
    (train_df["en"].str.strip() == train_df["vi"].str.strip())
    &
    (train_df["en"].str.contains(vi_pattern, na=False))
)

print("Remove mislabeled:", mislabeled_mask.sum())

train_df = train_df[
    ~mislabeled_mask
].reset_index(drop=True)


# 4. Convert lại Dataset
med_dataset = DatasetDict({
    "train": Dataset.from_pandas(train_df, preserve_index=False),
    "validation": Dataset.from_pandas(val_df, preserve_index=False),
    "test": Dataset.from_pandas(test_df, preserve_index=False),
})

print(med_dataset)

Remove train leakage: 20
Remove mislabeled: 0
DatasetDict({
    train: Dataset({
        features: ['en', 'vi'],
        num_rows: 340694
    })
    validation: Dataset({
        features: ['en', 'vi'],
        num_rows: 8939
    })
    test: Dataset({
        features: ['en', 'vi'],
        num_rows: 8958
    })
})
